<div style="padding: 20px; background: linear-gradient(90deg, #4b6cb7 0%, #182848 100%); border-radius: 10px; color: white;">
    <h1 style="color: white; border-bottom: none;">🤝 Module 8.4: Multi-Agent Patterns</h1>
    <p style="font-size: 1.2em; opacity: 0.9;">Routers, Sequential Specialists, and Parallel Executors.</p>
</div>

---

Complex knowledge tasks often require multiple specialized agents working together. We will explore three vital patterns:
1. **Router Agent**: Analyzes the question and sends it to the correct specialist.
2. **Sequential Agents**: A Researcher hands notes to a Writer, who hands a draft to a Validator.
3. **Parallel Agents**: Multiple retrievers search different databases simultaneously to save time.

In [1]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document
from dotenv import load_dotenv
import os
import concurrent.futures

os.environ['TOKENIZERS_PARALLELISM'] = 'false'
load_dotenv()

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Setup Tech KB
tech_docs = [
    Document(page_content="Python 3.12 introduces new type parameter syntax for generics."),
    Document(page_content="LangChain supports async execution natively."),
]
tech_vs = Chroma.from_documents(tech_docs, embeddings, collection_name="tech_kb_v2")

# Setup Biz KB
biz_docs = [
    Document(page_content="Q3 revenue grew 23% year-over-year."),
    Document(page_content="Customer acquisition cost decreased by 12%."),
]
biz_vs = Chroma.from_documents(biz_docs, embeddings, collection_name="biz_kb_v2")

print("Databases Initialized.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Databases Initialized.


## 1. The Router Agent Pattern

In [2]:
groq_api_key = os.environ.get("GROQ_API_KEY")

if groq_api_key:
    llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
    
    router_prompt = ChatPromptTemplate.from_template("""
    You are a routing agent. Decide which database to search.
    Options: 'tech_db', 'business_db', 'none'
    Answer EXACTLY with the option name.
    Question: {question}
    """)
    
    def router_agent(question: str):
        choice = (router_prompt | llm | StrOutputParser()).invoke({"question": question}).strip().lower()
        print(f"  🔀 Router routed to: {choice.upper()}")
        
        if 'tech_db' in choice:
            docs = tech_vs.similarity_search(question, k=1)
        elif 'business_db' in choice:
            docs = biz_vs.similarity_search(question, k=1)
        else:
            docs = []
            
        ctx = "\n".join(d.page_content for d in docs)
        gen_prompt = ChatPromptTemplate.from_template("Answer using context (if provided):\n{context}\nQ: {question}")
        return (gen_prompt | llm | StrOutputParser()).invoke({"context": ctx, "question": question})

    print("--- ROUTER TESTS ---")
    for q in ["What is new in Python 3.12?", "How did revenue do in Q3?", "What is the capital of France?"]:
        print(f"\nUser: {q}")
        ans = router_agent(q)
        print(f"Ans: {ans.strip()}")

--- ROUTER TESTS ---

User: What is new in Python 3.12?
  🔀 Router routed to: TECH_DB


Ans: Python 3.12 introduces new type parameter syntax for generics.

User: How did revenue do in Q3?


  🔀 Router routed to: BUSINESS_DB
Ans: Revenue grew 23% year-over-year in Q3.

User: What is the capital of France?


  🔀 Router routed to: NONE
Ans: The capital of France is Paris.


## 2. Sequential Agents (The Factory Line)

In [3]:
if groq_api_key:
    def researcher(question: str) -> str:
        docs = tech_vs.similarity_search(question, k=2)
        ctx = "\n".join(d.page_content for d in docs)
        p = ChatPromptTemplate.from_template("Extract raw facts from this context regarding the question.\nCtx: {context}\nQ: {question}")
        return (p | llm | StrOutputParser()).invoke({"context": ctx, "question": question})
        
    def writer(research_notes: str, question: str) -> str:
        p = ChatPromptTemplate.from_template("Write a highly professional answer using THESE notes.\nNotes: {notes}\nQ: {question}")
        return (p | llm | StrOutputParser()).invoke({"notes": research_notes, "question": question})
        
    def validator(draft: str) -> str:
        p = ChatPromptTemplate.from_template("Review this draft. Is it professional? Score 1-10.\nDraft: {draft}")
        return (p | llm | StrOutputParser()).invoke({"draft": draft})

    print("\n--- SEQUENTIAL PIPELINE ---")
    q = "Does Langchain support async?"
    print(f"User: {q}")
    
    notes = researcher(q)
    print(f"\n🔍 Researcher Output:\n{notes.strip()}")
    
    draft = writer(notes, q)
    print(f"\n✒️ Writer Output:\n{draft.strip()}")
    
    review = validator(draft)
    print(f"\n👮 Validator Output:\n{review.strip()}")


--- SEQUENTIAL PIPELINE ---
User: Does Langchain support async?



🔍 Researcher Output:
Yes, LangChain supports async execution natively.



✒️ Writer Output:
Yes, LangChain supports asynchronous execution natively. This means that users can leverage asynchronous programming to efficiently manage concurrent tasks and improve overall system performance, without requiring additional workarounds or modifications to the LangChain framework.



👮 Validator Output:
I would score this draft an 8 out of 10 in terms of professionalism. Here's a breakdown of the strengths and weaknesses:

Strengths:

1. **Clear statement**: The draft clearly states that LangChain supports asynchronous execution natively.
2. **Technical accuracy**: The statement is technically accurate and conveys the benefits of asynchronous execution.
3. **Concise language**: The language is concise and easy to understand.

Weaknesses:

1. **Lack of context**: The statement assumes that the reader is familiar with asynchronous programming and its benefits. Adding a brief explanation or context would make the statement more accessible to a wider audience.
2. **No specific examples**: While the statement mentions the benefits of asynchronous execution, it would be more convincing with specific examples or use cases.
3. **No conclusion or call-to-action**: The statement ends abruptly without summarizing the key point or encouraging the reader to explore further.

T

## 3. Parallel Retrieval (Concurrency)

In [4]:
if groq_api_key:
    def parallel_retrieve(question: str):
        print(f"\n--- PARALLEL RETRIEVAL ---")
        print(f"Querying multiple DBs concurrently for: '{question}'")
        
        # Uses Python threading to hit both vector stores at the exact same time
        with concurrent.futures.ThreadPoolExecutor() as executor:
            tech_future = executor.submit(tech_vs.similarity_search, question, 1)
            biz_future = executor.submit(biz_vs.similarity_search, question, 1)
            
            tech_docs = tech_future.result()
            biz_docs = biz_future.result()
            
        combined = tech_docs + biz_docs
        print(f"Total documents retrieved instantly: {len(combined)}")
        for i, d in enumerate(combined):
            print(f"  [{i+1}] {d.page_content}")
            
    parallel_retrieve("Show me both Python async features and Q3 revenue.")


--- PARALLEL RETRIEVAL ---
Querying multiple DBs concurrently for: 'Show me both Python async features and Q3 revenue.'
Total documents retrieved instantly: 2
  [1] LangChain supports async execution natively.
  [2] Q3 revenue grew 23% year-over-year.
